# Диагностика EMA и BatchNorm
Запускайте после окончания текущего обучения, в отдельном kernel. Сравниваем три варианта **одного** `best.pt` на одинаковых validation rows. BN пересчитывается только на train, без аугментаций и градиентов. Исходный запуск не изменяется.

Каждый вариант проходит полную валидацию; всего три прохода и короткая калибровка BN. По умолчанию калибровка использует 512 train-изображений. Это диагностика, а не гарантированное улучшение.

In [1]:
from pathlib import Path
import sys
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

RUN_DIR = root / "runs" / "efficientvit_b2_1024_mixed_original"
CHECKPOINT = "best.pt"
CALIBRATION_IMAGES = 512
BATCH_SIZE = 2
WORKERS = 4
DEVICE = "cuda"

In [2]:
import copy
import json
from dataclasses import fields, replace
from datetime import datetime

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.config import (ExperimentConfig, PathsConfig, ModelConfig, AugmentationConfig,
                        DatasetConfig, TrainConfig, EvalConfig)
from src.data.data_workspace import DataWorkspace
from src.data.dataset import AIIJCDataset
from src.data.collation import ValidationCollator
from src.eval.metadata import build_metadata
from src.eval.splits import make_stratified_val_folds, train_val_fold_split
from src.progress import ConsoleProgress
from src.training.builders import build_model, build_amp, DataLoaderThreadLimits
from src.training.runs import Run
from src.training.validation import validate

DataLoaderThreadLimits.apply()
checkpoint = torch.load(RUN_DIR / "ckpt" / CHECKPOINT, map_location="cpu", weights_only=False)
snapshot = checkpoint["cfg"]
if "model" in snapshot:
    config = ExperimentConfig.from_dict(snapshot)
else:
    sections = {"paths": PathsConfig, "model": ModelConfig, "augmentation": AugmentationConfig,
                "dataset": DatasetConfig, "train": TrainConfig, "eval": EvalConfig}
    nested = {name: {f.name: snapshot[f.name] for f in fields(kind) if f.name in snapshot}
              for name, kind in sections.items()}
    config = ExperimentConfig.from_dict({**nested, "seed": snapshot["seed"]})
config = replace(config, train=replace(config.train, device=DEVICE, batch_size=BATCH_SIZE, workers=WORKERS))
assert checkpoint.get("ema") is not None, "Checkpoint must contain EMA weights"
amp = build_amp(config.train)
print("Checkpoint epoch:", checkpoint["epoch"], "samples:", checkpoint.get("samples"))
print("Validation resolution:", config.eval.resolution)

Checkpoint epoch: 5 samples: 144000
Validation resolution: original


In [3]:
workspace = DataWorkspace(config.paths.data_path)
val_rows = Run.open(RUN_DIR).load_rows()
metadata = build_metadata(workspace, workers=WORKERS)
folded = make_stratified_val_folds(metadata, n_folds=config.dataset.n_folds, seed=config.seed)
train_rows, expected_val = train_val_fold_split(folded, config.dataset.fold)
assert set(val_rows.chng_img_path) == set(expected_val.chng_img_path), "Validation split changed"
assert not set(train_rows.group_id) & set(val_rows.group_id), "Train/val group leakage"
calibration_rows = train_rows.sample(n=min(CALIBRATION_IMAGES, len(train_rows)), random_state=config.seed)

def make_loader(rows, original_targets=False):
    dataset = AIIJCDataset(workspace, rows, False, config.dataset.image_size, config.seed,
                          mode="val", original_targets=original_targets,
                          resize_mode=config.dataset.resize_mode)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS,
                      persistent_workers=False, pin_memory=DEVICE.startswith("cuda"),
                      collate_fn=ValidationCollator(), worker_init_fn=DataLoaderThreadLimits.apply)

val_loader = make_loader(val_rows, original_targets=config.eval.resolution == "original")
calibration_loader = make_loader(calibration_rows)
print("Validation images:", len(val_rows), "BN calibration images (train only):", len(calibration_rows))

[14:03:22] Метаданные загружены из кеша: D:\Challenges\AIIJC2026\data\train_stage1\.cache\metadata_v1.parquet
Validation images: 20740 BN calibration images (train only): 512


In [4]:
@torch.no_grad()
def recalibrate_bn(model, loader, amp):
    # Enable only BN updates; dropout and all other layers stay in eval mode.
    model.eval()
    bn_layers = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)
                 and m.track_running_stats]
    momenta = {m: m.momentum for m in bn_layers}
    seen = 0
    for m in bn_layers:
        m.reset_running_stats()
        m.train()
    try:
        for batch in ConsoleProgress.iterate(loader, "BN calibration (train images)"):
            images = batch["image"].to(amp.device)
            fmap = batch["fmap"].to(amp.device)
            kwargs = {"valid_mask": batch["valid_mask"].to(amp.device)} if "valid_mask" in batch else {}
            batch_size = images.shape[0]
            for m in bn_layers:
                m.momentum = batch_size / (seen + batch_size)
            with amp.autocast():
                model(images, fmap, **kwargs)
            seen += batch_size
    finally:
        for m in bn_layers:
            m.momentum = momenta[m]
        model.eval()
    if not seen:
        raise ValueError("Calibration loader is empty")
    return len(bn_layers), seen

In [5]:
output_dir = RUN_DIR.parent / (RUN_DIR.name + "_bn_diagnostic_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
output_dir.mkdir(exist_ok=False)
results = []
fixed_point = None

for name, weights_key, calibrate in [("raw", "model", False), ("ema", "ema", False),
                                      ("ema_recalibrated_bn", "ema", True)]:
    print("Evaluating:", name)
    model = build_model(config.model, pretrained=False).to(amp.device, memory_format=torch.channels_last)
    model.load_state_dict(checkpoint[weights_key])
    try:
        if calibrate:
            print("BN layers, calibration images:", recalibrate_bn(model, calibration_loader, amp))
        result = validate(model, val_loader, amp, config, amp.device)
        result.accumulator.save(output_dir / (name + ".npz"))
        results.append({"variant": name, **result.tuned.as_dict()})
        if name == "ema":
            fixed_point = result.operating_point
        del result
    finally:
        del model
        if amp.device.type == "cuda":
            torch.cuda.empty_cache()

# Also compare at the same operating point, chosen for unchanged EMA.
from src.training.metric import AICAccumulator
for row in results:
    acc = AICAccumulator.load(output_dir / (row["variant"] + ".npz"))
    fixed = acc.evaluate(*fixed_point)
    row.update(aic_fixed=fixed.aic, dice_fixed=fixed.dice_pos, fpr_fixed=fixed.fpr_neg)
    del acc
report = pd.DataFrame(results)
report.to_csv(output_dir / "comparison.csv", index=False)
(output_dir / "diagnostic.json").write_text(json.dumps({
    "source_run": str(RUN_DIR), "checkpoint": CHECKPOINT, "epoch": checkpoint["epoch"],
    "samples": checkpoint.get("samples"), "calibration_images": len(calibration_rows),
    "seed": config.seed, "resolution": config.eval.resolution,
    "resize_mode": config.dataset.resize_mode, "fixed_point_from_ema": fixed_point,
}, indent=2), encoding="utf-8")
calibration_rows.to_parquet(output_dir / "calibration_rows.parquet", index=False)
print("Results:", output_dir)
report[["variant", "aic", "dice_pos", "fpr_neg", "aic_fixed", "dice_fixed", "fpr_fixed"]]

Evaluating: raw
[14:03:32] Валидация, батчи: начало, всего 10370; ожидание первого элемента
[14:03:52] Валидация, батчи: 1/10370, прошло 00:00:20
[14:04:22] Валидация, батчи: 606/10370, прошло 00:00:50, осталось ~00:08:04
[14:04:52] Валидация, батчи: 1169/10370, прошло 00:01:20, осталось ~00:07:53
[14:05:22] Валидация, батчи: 1691/10370, прошло 00:01:50, осталось ~00:07:42
[14:05:52] Валидация, батчи: 2201/10370, прошло 00:02:20, осталось ~00:07:26
[14:06:22] Валидация, батчи: 2680/10370, прошло 00:02:50, осталось ~00:07:11
[14:06:52] Валидация, батчи: 3191/10370, прошло 00:03:20, осталось ~00:06:45
[14:07:22] Валидация, батчи: 3689/10370, прошло 00:03:50, осталось ~00:06:21
[14:07:52] Валидация, батчи: 4214/10370, прошло 00:04:20, осталось ~00:05:51
[14:08:22] Валидация, батчи: 4820/10370, прошло 00:04:50, осталось ~00:05:11
[14:08:52] Валидация, батчи: 5419/10370, прошло 00:05:20, осталось ~00:04:34
[14:09:22] Валидация, батчи: 6026/10370, прошло 00:05:51, осталось ~00:03:58
[14:09:5

,variant,aic,dice_pos,fpr_neg,aic_fixed,dice_fixed,fpr_fixed
0,raw,0.866413,0.773713,0.015649,0.863696,0.777215,0.028169
1,ema,0.848001,0.753101,0.029734,0.848001,0.753101,0.029734
2,ema_recalibrated_bn,0.868456,0.790140,0.035994,0.855595,0.757554,0.017214


## Как интерпретировать
- **EMA после BN-калибровки лучше EMA**: свидетельство проблемы статистик BN в текущем режиме; проверить устойчивость на другой train-подвыборке/большем числе калибровочных изображений.
- **Raw лучше EMA**: EMA может отставать от текущей модели; это ещё не доказывает, что причина только в BN.
- **Все три близки и заметно хуже PVT**: вероятнее искать причину в архитектуре, разрешении и рецепте обучения.

`aic` использует отдельно подобранные пороги каждого варианта; `aic_fixed` — общие пороги исходной EMA. Сравниваются веса одной эпохи, а не независимо выбранные лучшие эпохи raw и EMA. Калибровка не меняет веса, но меняет BN-буферы только в памяти. Калиброванный checkpoint автоматически не сохраняется и не подменяет исходный.